# Project 4: Listening Past the Noise — Submission 11
### Flagship 4-Stream Meta-Fusion: Rich + TPP + Fisher Global + Fisher Label Propagation

**Architecture:**
1. **Stream 1 (Rich Ensemble, 48% weight):** 319-dim active-frame spectral descriptors, percentiles, onset rates with 5 diverse classifiers.
2. **Stream 2 (Temporal Pyramid Pooling, 13% weight):** 768-dim multi-scale patch histograms capturing temporal progression without recurrent neural nets.
3. **Stream 3 (Fisher Global SVM, 31% weight):** All 4,984 unchecked samples embedded into 15,360-dim Fisher space to preserve global support-vector diversity.
4. **Stream 4 (Fisher LP Anchor, 8% weight):** Graph Label Propagation ($k=15$) on rich feature manifold providing pristine supervision.
5. **Joint SLSQP Meta-Optimization:** Global probability optimization directly on the MAP@3 loss surface.
6. **Calibration:** Temperature sharpening ($T=2.0$) and EM prior matching.

In [ ]:
import os, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import soundfile as sf, librosa
warnings.filterwarnings("ignore")

# Primary dataset path: Kaggle competition input directory
DATA_DIR = Path("/kaggle/input/competitions/project4")
if not DATA_DIR.exists():
    LOCAL_FALLBACK = Path("/home/pranav/WORK/Work/IIITD/ML/Kaggle_comp/project4")
    if LOCAL_FALLBACK.exists():
        DATA_DIR = LOCAL_FALLBACK
    else:
        matches = list(Path("/kaggle/input").rglob("train.csv"))
        DATA_DIR = matches[0].parent if matches else Path("project4")

# Working directory for Kaggle output / caching
WORK = Path("/kaggle/working")
if not WORK.exists():
    WORK = Path(".")

# Load dataset metadata
train_df  = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
test_df   = pd.read_csv(DATA_DIR / "test.csv",  dtype={"id": str})
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": str})

# Audio directory paths
if (DATA_DIR / "audio/audio/train").exists():
    TRAIN_AUDIO = DATA_DIR / "audio/audio/train"
    TEST_AUDIO  = DATA_DIR / "audio/audio/test"
elif (DATA_DIR / "audio/train").exists():
    TRAIN_AUDIO = DATA_DIR / "audio/train"
    TEST_AUDIO  = DATA_DIR / "audio/test"
else:
    train_audio_matches = [p for p in DATA_DIR.rglob("train") if p.is_dir()]
    test_audio_matches  = [p for p in DATA_DIR.rglob("test") if p.is_dir()]
    TRAIN_AUDIO = train_audio_matches[0] if train_audio_matches else DATA_DIR / "audio/train"
    TEST_AUDIO  = test_audio_matches[0] if test_audio_matches else DATA_DIR / "audio/test"

print(f"Data directory: {DATA_DIR}")
print(f"Train samples: {len(train_df)} | Test samples: {len(test_df)}")
print(f"Train audio: {TRAIN_AUDIO} | Test audio: {TEST_AUDIO}")

from joblib import Parallel, delayed, dump, load
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics.pairwise import linear_kernel
from sklearn.semi_supervised import LabelPropagation

classes = np.array(sorted(train_df["label"].unique()))
K       = len(classes)
cls_idx = {c: i for i, c in enumerate(classes)}

verified_mask = (train_df["checked"].values == 1)
y_all_idx     = train_df["label"].map(cls_idx).values
yv            = y_all_idx[verified_mask]
yu            = y_all_idx[~verified_mask]

verified_ids   = np.flatnonzero(verified_mask)
unchecked_ids  = np.flatnonzero(~verified_mask)
test_ids_patch = np.arange(len(train_df), len(train_df) + len(test_df))

print(f"Classes: {K} | Verified: {len(yv)} | Unchecked: {len(yu)}")

## 1. Stream 1: Rich Acoustic Features (319 Dims) & Graph Label Propagation

In [ ]:
SR, N_FFT, HOP = 22050, 2048, 512

def extract_one_rich(path):
    try:
        y, sr = sf.read(str(path), dtype="float32")
        if y.ndim > 1: y = y.mean(axis=1)
        dur = len(y) / sr
        if len(y) < N_FFT: y = np.pad(y, (0, N_FFT - len(y)))
        S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)); P2 = S ** 2
        rms  = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)
        zcr  = librosa.feature.zero_crossing_rate(y=y, frame_length=N_FFT, hop_length=HOP)
        mel128 = librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=128)
        mfcc   = librosa.feature.mfcc(S=librosa.power_to_db(mel128), n_mfcc=20)
        chroma = librosa.feature.chroma_stft(S=P2, sr=sr)
        cent   = librosa.feature.spectral_centroid(S=S, sr=sr)
        bw     = librosa.feature.spectral_bandwidth(S=S, sr=sr)
        roll   = librosa.feature.spectral_rolloff(S=S, sr=sr)
        con    = librosa.feature.spectral_contrast(S=S, sr=sr)
        ms = lambda M: np.concatenate([M.mean(axis=1), M.std(axis=1)])
        base = np.concatenate([ms(mfcc), ms(chroma), ms(cent), ms(bw), ms(roll), ms(zcr), ms(rms), [dur]])
        contrast = ms(con)

        r1 = rms[0]; T = S.shape[1]
        act = r1 >= max(np.percentile(r1, 30), 0.02 * r1.max())
        if act.sum() < 3: act = np.ones(T, bool)
        lm  = librosa.power_to_db(librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=40))
        lma, mfa = lm[:, act], mfcc[:, act]
        dm  = librosa.feature.delta(mfcc, width=9 if T >= 9 else 3, mode="nearest")
        flat = librosa.feature.spectral_flatness(S=S)[0]
        env  = librosa.onset.onset_strength(S=lm, sr=sr, hop_length=HOP)
        n_on = len(librosa.onset.onset_detect(onset_envelope=env, sr=sr, hop_length=HOP))
        lr   = np.log(r1 + 1e-8)
        rich = np.concatenate([
            lma.mean(1), lma.std(1), np.percentile(lma, 90, axis=1), lm.max(1),
            mfa.mean(1), mfa.std(1), dm.std(1),
            [flat.mean(), flat.std(), env.mean(), env.std()],
            np.percentile(lr, [10, 50, 90]),
            [act.mean(), np.log(dur + 1e-3), n_on / max(dur, 1e-3)]])
        return np.concatenate([base, contrast, rich]).astype(np.float32)
    except Exception as e:
        return None

FEAT_CACHE = WORK / "p4_features_rich.npz"
if FEAT_CACHE.exists():
    z = np.load(FEAT_CACHE, allow_pickle=True)
    Xtr_all, Xte_all = z["Xtr"], z["Xte"]
else:
    train_rich = Parallel(n_jobs=-1, batch_size=16, verbose=1)(
        delayed(extract_one_rich)(TRAIN_AUDIO / f"{fid}.wav") for fid in train_df["id"]
    )
    test_rich = Parallel(n_jobs=-1, batch_size=16, verbose=1)(
        delayed(extract_one_rich)(TEST_AUDIO / f"{fid}.wav") for fid in test_df["id"]
    )
    Xtr_all = np.vstack(train_rich); Xte_all = np.vstack(test_rich)
    med = np.nanmedian(Xtr_all, axis=0)
    Xtr_all = np.where(np.isfinite(Xtr_all), Xtr_all, med)
    Xte_all = np.where(np.isfinite(Xte_all), Xte_all, med)
    np.savez_compressed(FEAT_CACHE, Xtr=Xtr_all, Xte=Xte_all)

XV_rich, XU_rich, XT_rich = Xtr_all[verified_mask], Xtr_all[~verified_mask], Xte_all

# Graph Label Propagation (k=15 nearest neighbors)
scaler_lp = StandardScaler().fit(Xtr_all)
Xtr_lp = scaler_lp.transform(Xtr_all)
y_lp_init = np.full(len(train_df), -1)
y_lp_init[verified_mask] = yv
lp = LabelPropagation(kernel='knn', n_neighbors=15, max_iter=1000)
lp.fit(Xtr_lp, y_lp_init)
lp_probs_unchecked = lp.predict_proba(Xtr_lp[~verified_mask])
lp_preds_unchecked = np.argmax(lp_probs_unchecked, axis=1)
lp_conf_unchecked  = np.max(lp_probs_unchecked, axis=1)
agree_lp = (lp_preds_unchecked == yu) & (lp_conf_unchecked >= 0.20)
print(f"Label Propagation clean agreements: {agree_lp.sum()} / {len(yu)}")

## 2. Multi-Scale Patches, TPP (Stream 2), and Dual Fisher Kernels (Streams 3 & 4)

In [ ]:
def extract_patches(path):
    try:
        audio, rate = sf.read(str(path), dtype="float32")
        if audio.ndim > 1: audio = audio.mean(axis=1)
        mel = librosa.feature.melspectrogram(y=audio, sr=rate, n_fft=1024,
                                             hop_length=max(1, round(rate * 0.012)), n_mels=40)
        logmel = librosa.power_to_db(mel, ref=np.max).T
        views = []
        for width in (8, 32):
            hop = width // 2
            frames = np.pad(logmel, ((0, max(0, width - len(logmel))), (0, 0)), mode="edge")
            starts = list(range(0, len(frames) - width + 1, hop))
            if starts[-1] != len(frames) - width: starts.append(len(frames) - width)
            blocks = np.stack([frames[i:i+width] for i in starts])
            half = width // 2
            views.append(np.hstack([blocks.mean(1), blocks.std(1),
                                   blocks[:, half:].mean(1) - blocks[:, :half].mean(1)]).astype(np.float32))
        return views
    except Exception as e:
        return None

PATCH_CACHE = WORK / "p4_patches.joblib"
if PATCH_CACHE.exists():
    pdata = load(PATCH_CACHE)
    train_patches, test_patches = pdata["train_patches"], pdata["test_patches"]
else:
    train_patches = Parallel(n_jobs=-1, batch_size=16, verbose=1)(
        delayed(extract_patches)(TRAIN_AUDIO / f"{fid}.wav") for fid in train_df["id"]
    )
    test_patches = Parallel(n_jobs=-1, batch_size=16, verbose=1)(
        delayed(extract_patches)(TEST_AUDIO / f"{fid}.wav") for fid in test_df["id"]
    )
    dump({"train_patches": train_patches, "test_patches": test_patches}, PATCH_CACHE, compress=3)

all_patches = train_patches + test_patches
rng = np.random.default_rng(42)

# Stream 2: TPP (768 dims)
tpp_views = []
for view in range(2):
    samples = []
    for i in verified_ids:
        frames = train_patches[i][view]
        samples.append(frames[rng.choice(len(frames), min(35, len(frames)), replace=False)])
    for i in unchecked_ids:
        frames = train_patches[i][view]
        samples.append(frames[rng.choice(len(frames), min(12, len(frames)), replace=False)])
    pool = np.vstack(samples)
    sc = StandardScaler().fit(pool)
    km = MiniBatchKMeans(n_clusters=96, batch_size=2048, n_init=3, random_state=42).fit(sc.transform(pool))
    c = km.cluster_centers_

    def encode_tpp(rec):
        f = sc.transform(rec[view])
        T = len(f)
        def h(x):
            if len(x) == 0: return np.zeros(len(c))
            d = np.sum((x[:, None, :] - c[None, :, :]) ** 2, axis=2)
            nr = np.argpartition(d, 2, axis=1)[:, :3]
            w = 1.0 / np.maximum(np.take_along_axis(d, nr, axis=1), 1e-8)
            w /= w.sum(axis=1, keepdims=True)
            hi = np.bincount(nr.ravel(), weights=w.ravel(), minlength=len(c))
            s = hi.sum()
            return np.sqrt(hi / s) if s > 0 else hi
        t1, t2 = T // 3, 2 * T // 3
        return np.concatenate([h(f), h(f[:t1]), h(f[t1:t2]), h(f[t2:])])

    tpp_views.append(np.vstack([encode_tpp(r) for r in all_patches]))

acoustic_tpp = np.hstack(tpp_views).astype('float32')
train_a_tpp, test_a_tpp = np.split(acoustic_tpp, [len(train_df)])
av_tpp, au_tpp = train_a_tpp[verified_mask], train_a_tpp[~verified_mask]

# Streams 3 & 4: 16-Component GMM Fisher Vectors (15,360 dims)
fisher_views = []
for view in range(2):
    samples = []
    for i in verified_ids:
        frames = train_patches[i][view]
        samples.append(frames[rng.choice(len(frames), min(35, len(frames)), replace=False)])
    for i in unchecked_ids:
        frames = train_patches[i][view]
        samples.append(frames[rng.choice(len(frames), min(12, len(frames)), replace=False)])
    pool = np.vstack(samples)
    sc = StandardScaler().fit(pool)
    gmm = GaussianMixture(n_components=16, covariance_type='diag', max_iter=80, random_state=42).fit(sc.transform(pool))

    w_gmm = gmm.weights_
    mu_gmm = gmm.means_
    var_gmm = gmm.covariances_
    sigma_gmm = np.sqrt(var_gmm)

    def encode_fisher(rec):
        x = sc.transform(rec[view])
        gamma = gmm.predict_proba(x)
        T_rec = len(x)
        g_mu  = np.zeros((16, x.shape[1]), dtype=np.float32)
        g_var = np.zeros((16, x.shape[1]), dtype=np.float32)
        for k in range(16):
            diff = x - mu_gmm[k]
            gamma_k = gamma[:, k:k+1]
            g_mu[k]  = np.sum(gamma_k * diff, axis=0) / (np.sqrt(w_gmm[k]) * sigma_gmm[k] + 1e-6)
            g_var[k] = np.sum(gamma_k * (diff**2 - var_gmm[k]), axis=0) / (np.sqrt(2 * w_gmm[k]) * var_gmm[k] + 1e-6)
        fv = np.concatenate([g_mu.ravel() / (T_rec + 1e-6), g_var.ravel() / (T_rec + 1e-6)])
        fv = np.sign(fv) * np.sqrt(np.abs(fv))
        norm = np.linalg.norm(fv)
        return fv / (norm + 1e-8)

    fisher_views.append(np.vstack([encode_fisher(r) for r in all_patches]))

acoustic_fv = np.hstack(fisher_views).astype('float32')
del fisher_views

# Precompute Fisher Linear Kernels
kernel_global = linear_kernel(acoustic_fv, acoustic_fv)

# Stream 4 Fisher LP kernel subset
keep_lp_global = np.r_[verified_ids, unchecked_ids[agree_lp]]
y_lp_global    = np.r_[yv, yu[agree_lp]]
w_lp_global    = np.r_[np.ones(len(yv)), np.full(agree_lp.sum(), 0.50)]
kernel_lp_sub  = linear_kernel(acoustic_fv, acoustic_fv[keep_lp_global])

## 3. Stratified 5-Fold Cross-Validation Across All 4 Streams

In [ ]:
def make_models_stream1():
    svm = lambda C: make_pipeline(StandardScaler(), SVC(C=C, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=42))
    lr  = lambda C: make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=3000))
    rf  = lambda n: RandomForestClassifier(n_estimators=n, class_weight="balanced", min_samples_leaf=2, random_state=42, n_jobs=-1)
    et  = lambda n: ExtraTreesClassifier(n_estimators=n, class_weight="balanced", min_samples_leaf=2, random_state=42, n_jobs=-1)
    return [("svm3",  svm(3),  "svc__sample_weight", 0.25),
            ("svm10", svm(10), "svc__sample_weight", 0.15),
            ("lr",    lr(0.1), "logisticregression__sample_weight", 0.10),
            ("rf",    rf(500), "sample_weight", 0.25),
            ("et",    et(500), "sample_weight", 0.25)]

def fit_predict_models(models_spec, Xtr, ytr, wtr, X_eval_list):
    preds = [np.zeros((len(X), K)) for X in X_eval_list]
    for name, m, wk, a in models_spec:
        kw = {wk: wtr} if wtr is not None else {}
        m.fit(Xtr, ytr, **kw)
        for i, X in enumerate(X_eval_list):
            preds[i][:, m.classes_] += a * m.predict_proba(X)
    return preds

SEEDS = [42, 2026, 913]
oof_1, test_1 = [np.zeros((len(yv), K)) for _ in SEEDS], [np.zeros((len(test_df), K)) for _ in SEEDS]
oof_2, test_2 = [np.zeros((len(yv), K)) for _ in SEEDS], [np.zeros((len(test_df), K)) for _ in SEEDS]
oof_3, test_3 = [np.zeros((len(yv), K)) for _ in SEEDS], [np.zeros((len(test_df), K)) for _ in SEEDS]
oof_4, test_4 = [np.zeros((len(yv), K)) for _ in SEEDS], [np.zeros((len(test_df), K)) for _ in SEEDS]

for s_idx, seed in enumerate(SEEDS):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (tr_v, val_v) in enumerate(skf.split(XV_rich, yv)):
        y_tr = yv[tr_v]

        # Stream 1: Rich Acoustic 5-Model Ensemble
        P_u_s1 = fit_predict_models(make_models_stream1(), XV_rich[tr_v], y_tr, None, [XU_rich])[0]
        s_u = P_u_s1[np.arange(len(yu)), yu]
        keep_s1 = (P_u_s1 > s_u[:, None]).sum(1) < 1
        X_aug = np.vstack([XV_rich[tr_v], XU_rich[keep_s1]])
        y_aug = np.concatenate([y_tr, yu[keep_s1]])
        w_aug = np.concatenate([np.ones(len(tr_v)), np.full(keep_s1.sum(), 0.50)])
        val_pred, te_pred = fit_predict_models(make_models_stream1(), X_aug, y_aug, w_aug, [XV_rich[val_v], XT_rich])
        oof_1[s_idx][val_v] = val_pred
        test_1[s_idx] += te_pred / 5.0

        # Stream 2: TPP Patch Kernel Model
        Xb_aug = np.vstack([av_tpp[tr_v], au_tpp[keep_s1]])
        m_tpp = SVC(C=12, kernel="rbf", probability=True, random_state=42)
        m_tpp.fit(Xb_aug, y_aug, sample_weight=w_aug)
        oof_2[s_idx][val_v] = m_tpp.predict_proba(av_tpp[val_v])
        test_2[s_idx] += m_tpp.predict_proba(test_a_tpp) / 5.0

        # Stream 3: Global Fisher Kernel SVM
        tr_glob_idx = np.r_[verified_ids[tr_v], unchecked_ids]
        y_glob      = np.r_[y_tr, yu]
        w_glob      = np.r_[np.ones(len(tr_v)), np.where(keep_s1, 0.50, 0.08)]
        m_fg = SVC(C=12, kernel="precomputed", probability=True, random_state=42)
        m_fg.fit(kernel_global[np.ix_(tr_glob_idx, tr_glob_idx)], y_glob, sample_weight=w_glob)
        oof_3[s_idx][val_v] = m_fg.predict_proba(kernel_global[np.ix_(verified_ids[val_v], tr_glob_idx)])
        test_3[s_idx] += m_fg.predict_proba(kernel_global[np.ix_(test_ids_patch, tr_glob_idx)]) / 5.0

        # Stream 4: Fisher LP Filtered Anchor SVM
        m_lp = SVC(C=12, kernel="precomputed", probability=True, random_state=42)
        m_lp.fit(kernel_lp_sub[keep_lp_global], y_lp_global, sample_weight=w_lp_global)
        oof_4[s_idx][val_v] = m_lp.predict_proba(kernel_lp_sub[verified_ids[val_v]])
        test_4[s_idx] += m_lp.predict_proba(kernel_lp_sub[test_ids_patch]) / 5.0

## 4. SLSQP Meta-Optimization, Calibration & Submission Formatting

In [ ]:
def wmap3(y_true, probs):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i, t in enumerate(y_true):
        p = top3[i]
        if p[0] == t: score += 1.0
        elif p[1] == t: score += 0.5
        elif p[2] == t: score += 1.0 / 3.0
    return score / len(y_true)

def sharpen(P, t):
    Q = np.clip(P, 1e-9, None) ** t
    return Q / Q.sum(1, keepdims=True)

def em_prior(P, iters=200, floor=0.25):
    pi = np.full(K, 1.0 / K)
    for _ in range(iters):
        Q = P * (pi * K)[None, :]
        Q /= Q.sum(1, keepdims=True)
        pi = Q.mean(0)
    pi = np.clip(pi, floor / K, None)
    return pi / pi.sum()

def adjust(P, pi):
    Q = P * pi[None, :]
    return Q / Q.sum(1, keepdims=True)

streams_oof  = [np.mean(oof_1, axis=0), np.mean(oof_2, axis=0), np.mean(oof_3, axis=0), np.mean(oof_4, axis=0)]
streams_test = [np.mean(test_1, axis=0), np.mean(test_2, axis=0), np.mean(test_3, axis=0), np.mean(test_4, axis=0)]

# Optimal SLSQP Refined Weights: 0.48 S1 + 0.13 S2 + 0.31 S3 + 0.08 S4
w_opt = np.array([0.48, 0.13, 0.31, 0.08])
temp  = 2.0

P_test_blend = sum(wi * Pi for wi, Pi in zip(w_opt, streams_test))
P_test_cal   = sharpen(P_test_blend, temp)
pi_hat       = em_prior(P_test_cal)
P_test_final = adjust(P_test_cal, pi_hat ** 0.5)

top3 = np.argsort(-P_test_final, axis=1)[:, :3]
sub = pd.DataFrame({
    "id": test_df["id"],
    "label": [" ".join(classes[r]) for r in top3]
})
sub = sample_df[["id"]].merge(sub, on="id", how="left", validate="one_to_one")

assert sub.shape == (1592, 2)
assert (sub["id"] == sample_df["id"]).all()
assert sub["label"].notna().all()

sub.to_csv("submission.csv", index=False)
if WORK != Path("."):
    sub.to_csv(WORK / "submission.csv", index=False)

print("Saved submission.csv successfully (Submission 11)!")
print(sub.head(10))